In [55]:
# --------------------------------------------------------------------------------
# 📦 Cell 1 – Install / Imports
# --------------------------------------------------------------------------------
# If you're in a fresh environment you may need to uncomment the pip lines once.
# !pip install torchvision torch tqdm albumentations==1.4.3 opencv-python-headless

import os
import random
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import cv2

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models

import albumentations as A
from albumentations.pytorch import ToTensorV2
from matplotlib import pyplot as plt
from tqdm.auto import tqdm


In [56]:
# --------------------------------------------------------------------------------
# 📂 Cell 2 – Paths & Hyper-parameters  (tuned)
# --------------------------------------------------------------------------------
DATA_CSV   = r"C:\Users\yozev\PycharmProjects\finetuning\combined_shape16_scores.csv"
IMG_DIR    = Path(r"C:\Users\yozev\OneDrive\Desktop\Shapes2\shape16")
REF_PATH   = IMG_DIR / "original16.png"

# Training setup
BATCH_SIZE     = 32
NUM_EPOCHS     = 30
LR             = 2e-4        # ← lower
WEIGHT_DECAY   = 1e-4
VAL_FRAC       = 0.15
RANDOM_SEED    = 42
UNFREEZE_EPOCH = 5           # unfroze backbone after this epoch
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


Device: cpu


In [57]:
# --------------------------------------------------------------------------------
# 🖼️ Cell 3 – Utilities + richer augmentations
# --------------------------------------------------------------------------------
def dilate_white_line(img: np.ndarray, ksize: int = 3, iterations: int = 1):
    kernel = np.ones((ksize, ksize), np.uint8)
    return cv2.dilate(img, kernel, iterations=iterations)

import torchvision.transforms as T

class BoldLine(T.RandomApply):
    """Wraps the dilate_white_line util into a torchvision transform."""
    def __init__(self, p=0.5):
        fn = lambda img: Image.fromarray(
            dilate_white_line(np.array(img), 3, 1), mode="L"
        )
        super().__init__([T.Lambda(fn)], p=p)

child_train_tf = T.Compose([
    T.RandomAffine(degrees=10, translate=(0.08,0.08), scale=(0.9,1.1), fill=0),
    T.RandomPerspective(distortion_scale=0.05, p=0.3),
    BoldLine(p=0.6),
    T.ToTensor(),
    T.RandomErasing(scale=(0.01,0.03), ratio=(0.3,3.3), value=0, p=0.4),
    T.Normalize([0.5],[0.5]),
])

child_val_tf = T.Compose([
    T.ToTensor(),
    T.Normalize([0.5],[0.5]),
])
ref_tf = child_val_tf


In [58]:
# --------------------------------------------------------------------------------
# 🗄️ Cell 4 – Dataset (skip rows with missing image files)
# --------------------------------------------------------------------------------
import torchvision.transforms as T

class BoldLine:
    def __init__(self, ksize=3, iterations=1, p=0.5):
        self.ksize, self.iter, self.p = ksize, iterations, p
    def __call__(self, img):
        if random.random() > self.p:
            return img
        arr     = np.array(img, dtype=np.uint8)
        kernel  = np.ones((self.ksize, self.ksize), np.uint8)
        dilated = cv2.dilate(arr, kernel, iterations=self.iter)
        return Image.fromarray(dilated, mode="L")

child_train_tf = T.Compose([
    T.RandomAffine(degrees=5, translate=(0.05,0.05), scale=(0.95,1.05), fill=0),
    BoldLine(p=0.5),
    T.ToTensor(), T.Normalize([0.5],[0.5]),
])
child_val_tf = T.Compose([
    T.ToTensor(), T.Normalize([0.5],[0.5]),
])
ref_tf = child_val_tf


class Shape16Dataset(Dataset):
    """
    Builds the filename from `child_id` and **drops rows whose image file
    does not exist**, so training never crashes on FileNotFoundError.
    """
    NAME_TEMPLATE = "{id}.png"   # edit if your naming differs

    def __init__(self, csv_path, img_dir, ref_path):
        df        = pd.read_csv(csv_path)
        self.img_dir  = Path(img_dir)
        self.ref_path = Path(ref_path)

        assert "child_id" in df.columns, "CSV missing 'child_id'"
        assert "score"    in df.columns, "CSV missing 'score'"

        # -------------------------------------------------------------
        # Keep only rows whose <img_dir>/<NAME_TEMPLATE> file exists
        # -------------------------------------------------------------
        df["img_path"] = df["child_id"].apply(
            lambda cid: self.img_dir / self.NAME_TEMPLATE.format(id=cid)
        )
        missing_mask   = ~df["img_path"].apply(Path.exists)
        n_missing      = missing_mask.sum()
        if n_missing:
            print(f"⚠️  Skipping {n_missing} rows with missing images.")
        self.data = df[~missing_mask].reset_index(drop=True)

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        row        = self.data.iloc[idx]
        img_path   = row["img_path"]
        score      = int(row["score"]) - 1        # 1-7 → 0-6

        child_img  = Image.open(img_path).convert("L")
        ref_img    = Image.open(self.ref_path).convert("L")

        tf = child_val_tf if getattr(self, "is_val", False) else child_train_tf
        child_tensor = tf(child_img)
        ref_tensor   = ref_tf(ref_img)

        return {
            "child":     child_tensor,
            "reference": ref_tensor,
            "label":     torch.tensor(score, dtype=torch.long),
        }

In [59]:
# --------------------------------------------------------------------------------
# 🧹 Cell 5 – Split dataset & build loaders  (num_workers = 0 for safety)
# --------------------------------------------------------------------------------
VAL_FRAC    = 0.15
BATCH_SIZE  = 32
RANDOM_SEED = 42

full_ds = Shape16Dataset(DATA_CSV, IMG_DIR, REF_PATH)

val_sz   = int(len(full_ds) * VAL_FRAC)
train_sz = len(full_ds) - val_sz

train_ds, val_ds = random_split(
    full_ds,
    [train_sz, val_sz],
    generator=torch.Generator().manual_seed(RANDOM_SEED),
)

train_ds.dataset.is_val = False
val_ds.dataset.is_val   = True

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0)

print(f"train imgs: {len(train_ds)} | val imgs: {len(val_ds)}")


⚠️  Skipping 2 rows with missing images.
train imgs: 762 | val imgs: 134


In [60]:
# --------------------------------------------------------------------------------
# 🧠 Cell 6 – Siamese ResNet-18 with staged unfreezing
# --------------------------------------------------------------------------------
class SiameseResNet(nn.Module):
    def __init__(self, backbone_name="resnet18", num_classes=7, freeze=True):
        super().__init__()
        backbone = models.__dict__[backbone_name](weights="IMAGENET1K_V1")
        backbone.conv1 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])  # 512-D
        # freeze?
        for p in self.backbone.parameters():
            p.requires_grad = not freeze
        self.classifier = nn.Sequential(
            nn.Linear(512*2, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )

    def forward(self, child, reference):
        f1 = self.backbone(child).flatten(1)
        f2 = self.backbone(reference).flatten(1)
        return self.classifier(torch.cat([f1, f2], 1))

model = SiameseResNet(freeze=True).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)


In [61]:
# --------------------------------------------------------------------------------
# 🔁 Cell 7 – Training & validation helper functions
# --------------------------------------------------------------------------------
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss, correct, total = 0, 0, 0
    for batch in tqdm(loader, leave=False):
        child  = batch["child"].to(DEVICE)
        ref    = batch["reference"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(child, ref)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total   += labels.size(0)
    return running_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0, 0, 0
    for batch in loader:
        child  = batch["child"].to(DEVICE)
        ref    = batch["reference"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        outputs = model(child, ref)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * labels.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total   += labels.size(0)
    return running_loss / total, correct / total


In [62]:
# --------------------------------------------------------------------------------
# 🚀 Cell 8 – Training loop with staged unfreeze & LR scheduler
# --------------------------------------------------------------------------------
history = {"train_loss": [], "val_loss": [],
           "train_acc": [],  "val_acc": []}

best_val = 0.0
for epoch in range(1, NUM_EPOCHS+1):

    # unfreeze backbone once head has stabilised
    if epoch == UNFREEZE_EPOCH:
        for p in model.backbone.parameters():
            p.requires_grad = True
        # re-build optimizer to include backbone params
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=NUM_EPOCHS - UNFREEZE_EPOCH + 1
        )
        print(f"🔓  Unfroze backbone at epoch {epoch}")

    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    vl_loss, vl_acc = evaluate(model, val_loader, criterion)
    scheduler.step()

    history["train_loss"].append(tr_loss); history["val_loss"].append(vl_loss)
    history["train_acc"].append(tr_acc);   history["val_acc"].append(vl_acc)

    is_best = vl_acc > best_val
    best_val = max(best_val, vl_acc)

    print(f"Epoch {epoch:02}/{NUM_EPOCHS} | "
          f"Train {tr_loss:.3f}/{tr_acc:.3f} | "
          f"Val {vl_loss:.3f}/{vl_acc:.3f} | "
          f"LR {scheduler.get_last_lr()[0]:.2e}"
          + ("  ✅ best so far" if is_best else ""))


  0%|          | 0/24 [00:00<?, ?it/s]

Epoch 01/30 | Train 1.841/0.214 | Val 1.869/0.157 | LR 1.99e-04  ✅ best so far


  0%|          | 0/24 [00:00<?, ?it/s]

Epoch 02/30 | Train 1.783/0.238 | Val 1.824/0.187 | LR 1.98e-04  ✅ best so far


  0%|          | 0/24 [00:00<?, ?it/s]

...

Epoch 18/30 | Train 0.036/0.991 | Val 3.094/0.321 | LR 8.79e-05


  0%|          | 0/24 [00:00<?, ?it/s]

Epoch 19/30 | Train 0.025/0.996 | Val 4.007/0.261 | LR 7.61e-05


  0%|          | 0/24 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# --------------------------------------------------------------------------------
# 📈 Cell 9 – Plot learning curves
# --------------------------------------------------------------------------------
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(history["train_loss"], label="train")
plt.plot(history["val_loss"],   label="val")
plt.ylabel("Loss"); plt.xlabel("Epoch"); plt.legend(); plt.title("Loss")

plt.subplot(1,2,2)
plt.plot(history["train_acc"], label="train")
plt.plot(history["val_acc"],   label="val")
plt.ylabel("Accuracy"); plt.xlabel("Epoch"); plt.legend(); plt.title("Accuracy")
plt.tight_layout()
plt.show()


In [ ]:
# --------------------------------------------------------------------------------
# 💾 Cell 10 – Save the trained model
# --------------------------------------------------------------------------------
MODEL_PATH = "shape16_similarity_model.pt"
torch.save(model.state_dict(), MODEL_PATH)
print("Model saved to", MODEL_PATH)
